# Search Ranking Algorithm Explanation

This notebook explains how the search ranking algorithm works with real data examples.

In [5]:
import pandas as pd
import sqlite3
import json
import requests
from IPython.display import display, HTML

# Configuration
DB_PATH = 'var/stage1.db'
ES_URL = 'http://localhost:9200'
ES_INDEX = 'products'

## 1. BM25 Text Matching

BM25 is the core text relevance algorithm. It scores documents based on term frequency and inverse document frequency.

**Formula:**
```
BM25(D, Q) = Σ IDF(qi) * (f(qi, D) * (k1 + 1)) / (f(qi, D) + k1 * (1 - b + b * |D|/avgdl))
```

Where:
- `f(qi, D)` = term frequency in document
- `IDF(qi)` = log(N / n(qi)) where N = total docs, n(qi) = docs containing term
- `k1` = term saturation parameter (default 1.2)
- `b` = length normalization (default 0.75)
- `|D|` = document length, `avgdl` = average document length

In [6]:
# BM25 Query with field boosting
def search_bm25(query, size=10):
    """Search using BM25 with field boosting."""
    es_query = {
        "size": size,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^3", "brand^2", "category_path^1.5", "description"],
                "type": "best_fields"
            }
        }
    }
    
    response = requests.post(
        f"{ES_URL}/{ES_INDEX}/_search",
        headers={"Content-Type": "application/json"},
        json=es_query
    )
    
    return response.json()

# Example search
result = search_bm25("bluetooth speaker", size=5)
print(f"Total hits: {result['hits']['total']['value']}")
print("\nTop 5 BM25 results:")
for hit in result['hits']['hits']:
    src = hit['_source']
    print(f"  - {src['title'][:60]}... (score: {hit['_score']:.2f})")

Total hits: 3499

Top 5 BM25 results:
  - Bamboo Speaker - Portable Bluetooth Speaker, Brown... (score: 31.46)
  - Braxus Solar Portable Bluetooth Speaker | Waterproof Speaker... (score: 30.77)
  - iHome Iron Man Bluetooth Speaker, Wireless Speaker with Rech... (score: 29.42)
  - Inossa Conduction Speaker, Bluetooth Mini Speaker, Turn Any ... (score: 29.23)
  - Microlab H21 Bluetooth Wireless Speaker... (score: 29.09)


### Field Boosting Explanation

Fields are boosted to prioritize matches in important fields:
- `title^3` - Title matches are 3x more important
- `brand^2` - Brand matches are 2x more important
- `category_path^1.5` - Category matches are 1.5x more important
- `description^1` - Description matches have normal weight

## 2. Weighted Re-ranking

After BM25, results are re-ranked using product statistics.

**Final Score Formula:**
```
final_score = 0.40 * normalized_bm25 + 
              0.25 * normalized_popularity + 
              0.15 * normalized_conversion + 
              0.10 * normalized_review + 
              0.10 * in_stock_boost
```

**Normalization:**
- `normalized_score = (score - min) / (max - min)` for all scores 0-1 range

In [7]:
# Get product stats from database
def get_product_stats(product_ids):
    """Fetch product statistics from database."""
    conn = sqlite3.connect(DB_PATH)
    placeholders = ','.join(['?' for _ in product_ids])
    query = f"""
        SELECT product_id, view_count, ctr_proxy, conversion_rate, 
               review_score, review_count, popularity_score
        FROM product_stats 
        WHERE product_id IN ({placeholders})
    """
    df = pd.read_sql_query(query, conn, params=product_ids)
    conn.close()
    return df

# Calculate normalized scores
def normalize(series):
    """Min-max normalization to 0-1 range."""
    if series.max() == series.min():
        return series * 0  # All same value -> 0
    return (series - series.min()) / (series.max() - series.min())

# Example: Get stats for top BM25 results
if result['hits']['hits']:
    product_ids = [hit['_source']['product_id'] for hit in result['hits']['hits']]
    stats_df = get_product_stats(product_ids)
    print("Product stats retrieved:")
    display(stats_df)

Product stats retrieved:


,product_id,view_count,ctr_proxy,conversion_rate,review_score,review_count,popularity_score
0,B0097OUB18,0,0.0,1.000000,None,0,3.044522
1,B01E99W190,3,0.0,0.333333,None,0,2.197225
2,B0976J7M2L,2,0.0,0.500000,None,0,2.079442
3,B09YSVB816,1,0.0,1.000000,None,0,1.945910
4,B0B7KX5NZ8,0,0.0,1.000000,None,0,2.397895


In [4]:
# Calculate weighted scores
def calculate_weighted_score(bm25_score, popularity, conversion, review, in_stock=True):
    """Calculate final weighted score."""
    # Weights
    W_BM25 = 0.40
    W_POPULARITY = 0.25
    W_CONVERSION = 0.15
    W_REVIEW = 0.10
    W_INSTOCK = 0.10
    
    # In-stock boost (default True for prototype)
    instock_boost = 0.5 if in_stock else 0.0
    
    final_score = (
        W_BM25 * bm25_score +
        W_POPULARITY * popularity +
        W_CONVERSION * conversion +
        W_REVIEW * review +
        W_INSTOCK * instock_boost
    )
    
    return final_score

print("Weighted scoring formula:")
print("final_score = 0.40*BM25 + 0.25*Popularity + 0.15*Conversion + 0.10*Review + 0.10*InStock")

Weighted scoring formula:
final_score = 0.40*BM25 + 0.25*Popularity + 0.15*Conversion + 0.10*Review + 0.10*InStock


## 3. Personalization Boost

User category affinity boosts products in categories they prefer.

**Boost Formula:**
```
personalized_score = base_score + (affinity / 10) * personalization_weight
```

Where `personalization_weight` is configurable (default 0.3)

In [ ]:
# Get user category affinity
def get_user_affinity(user_id):
    """Fetch user category affinities."""
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT category_path, affinity_score
        FROM user_category_affinity
        WHERE user_id = ?
        ORDER BY affinity_score DESC
        LIMIT 5
    """
    df = pd.read_sql_query(query, conn, params=[user_id])
    conn.close()
    return df

# Find a user with affinity data
conn = sqlite3.connect(DB_PATH)
sample_user = pd.read_sql_query(
    "SELECT DISTINCT user_id FROM user_category_affinity LIMIT 1", conn
).iloc[0]['user_id']
conn.close()

print(f"Sample user: {sample_user}")
affinity_df = get_user_affinity(sample_user)
print("\nUser's top category affinities:")
display(affinity_df)

In [ ]:
# Apply personalization boost
def apply_personalization(products, user_affinities, weight=0.3):
    """Apply personalization boost to products."""
    boosted_products = []
    
    for product in products:
        score = product['score']
        category = product.get('category_path', '')
        
        # Find matching affinity
        for _, row in user_affinities.iterrows():
            aff_category = row['category_path']
            affinity = row['affinity_score']
            
            # Exact or parent category match
            if category == aff_category or category.startswith(aff_category + '/'):
                boost = (affinity / 10) * weight
                score += boost
                product['personalization_boost'] = boost
                break
        
        product['final_score'] = score
        boosted_products.append(product)
    
    # Sort by final score
    boosted_products.sort(key=lambda x: x['final_score'], reverse=True)
    return boosted_products

print("Personalization boost applied based on category affinity")

## 4. Semantic Fallback (ChromaDB)

When BM25 returns poor results, semantic search provides a fallback.

**Trigger Conditions:**
- `max_bm25_score < 5.0` (weak text match)
- `results_count < size/2` (insufficient results)

Semantic search uses vector embeddings to find conceptually similar products.

In [ ]:
# Check ChromaDB status
def check_chromadb():
    """Check if ChromaDB is available."""
    try:
        import chromadb
        client = chromadb.PersistentClient(path='./var/chroma')
        collection = client.get_collection('product_embeddings')
        count = collection.count()
        return True, count
    except Exception as e:
        return False, str(e)

available, info = check_chromadb()
if available:
    print(f"ChromaDB available with {info} embeddings")
else:
    print(f"ChromaDB not available: {info}")

## 5. Complete End-to-End Example

Let's trace through a complete search for "bluetooth speaker"

In [ ]:
def full_search_trace(query, user_id=None):
    """Complete search with step-by-step explanation."""
    print(f"\n{'='*60}")
    print(f"SEARCH: '{query}'")
    if user_id:
        print(f"USER: {user_id}")
    print('='*60)
    
    # Step 1: BM25 Search
    print("\nStep 1: BM25 Text Search")
    print("-" * 40)
    result = search_bm25(query, size=10)
    
    hits = result['hits']['hits']
    max_score = max(h['_score'] for h in hits) if hits else 0
    print(f"Found {len(hits)} results, max score: {max_score:.2f}")
    
    # Step 2: Get Product Stats
    print("\nStep 2: Fetch Product Stats")
    print("-" * 40)
    product_ids = [h['_source']['product_id'] for h in hits]
    stats_df = get_product_stats(product_ids)
    
    # Create lookup
    stats_lookup = dict(zip(stats_df['product_id'], stats_df.to_dict('records')))
    
    # Step 3: Calculate Weighted Scores
    print("\nStep 3: Calculate Weighted Scores")
    print("-" * 40)
    
    products = []
    for hit in hits:
        src = hit['_source']
        pid = src['product_id']
        stats = stats_lookup.get(pid, {})
        
        products.append({
            'product_id': pid,
            'title': src['title'][:50],
            'category': src.get('category_path', '')[:30],
            'bm25_score': hit['_score'],
            'popularity': stats.get('popularity_score', 0),
            'conversion': stats.get('conversion_rate', 0),
            'review': stats.get('review_score', 3.5)
        })
    
    # Normalize scores
    if products:
        df = pd.DataFrame(products)
        df['bm25_norm'] = normalize(df['bm25_score'])
        df['pop_norm'] = normalize(df['popularity'].fillna(0))
        df['conv_norm'] = normalize(df['conversion'].fillna(0))
        df['rev_norm'] = normalize(df['review'].fillna(3.5))
        
        df['weighted_score'] = (
            0.40 * df['bm25_norm'] +
            0.25 * df['pop_norm'] +
            0.15 * df['conv_norm'] +
            0.10 * df['rev_norm'] +
            0.10 * 0.5  # in_stock boost
        )
        
        print("\nWeighted score breakdown:")
        print(df[['title', 'bm25_score', 'popularity', 'weighted_score']].to_string())
    
    # Step 4: Apply Personalization
    if user_id and products:
        print("\nStep 4: Apply Personalization")
        print("-" * 40)
        affinity = get_user_affinity(user_id)
        if not affinity.empty:
            print(f"User has {len(affinity)} category affinities")
            # Apply boost logic here...
        else:
            print("No affinity data - using base scores")
    
    # Step 5: Final Results
    print("\nStep 5: Final Ranked Results")
    print("-" * 40)
    if products:
        df = df.sort_values('weighted_score', ascending=False)
        display(df[['title', 'category', 'weighted_score']].head(10))
    
    return products

# Run example
results = full_search_trace("bluetooth speaker")

## Summary

The search ranking algorithm combines:

1. **BM25 Text Matching (40%)** - Core relevance based on query terms
2. **Popularity Score (25%)** - Product engagement metrics
3. **Conversion Rate (15%)** - Purchase conversion efficiency
4. **Review Score (10%)** - Customer satisfaction indicator
5. **In-Stock Bonus (10%)** - Availability preference
6. **Personalization (+bonus)** - User category affinity boost

This multi-factor approach ensures relevant, popular, and personalized results.